# DAT535 Project – Group 24: Norgespris and electricity use
## Notebook 01: Scope, Dataset and Bronze layer

This notebook implements the first three sections of our *Data Engineering Project Definition Guide (v3)*:

| Section | What this notebook does |
|---|---|
| 1. Project statement | Writes down the scope we build against |
| 2. Dataset | Describes the sources, then **profiles the real files with Spark** (section 4 of this notebook) |
| 3. Bronze layer | Downloads / copies raw files **unchanged**, writes a manifest with checksums, skips duplicates, flags schema changes |

**How this notebook fits the repo**

* The same ingestion code is in `bronze_pipeline.py` (generated from the code cells in sections 3.1–3.9), so GitHub Actions can
  run it without Jupyter: `python run_pipeline.py bronze`.
* Dev and prod are separated by two environment variables, set by the workflows in `.github/workflows/`:
  `DAT535_ENV` (`dev`/`prod`) and `DAT535_OUTPUT_DIR` (`~/dat535-data/dev` or `~/dat535-data/prod`).
  When you run the notebook yourself it defaults to **dev**.
* Tests for the Bronze code: `pytest -q tests/`.

**Section 5 lists problems we found in the definition and in the data – read it before starting Silver.**

---
# 1. Project statement (scope)

**What we build:** a batch pipeline in PySpark with Bronze, Silver and Gold layers that reads CSV files from
Elhub (electricity), MET/Seklima (temperature) and SSB (population).

**Question it answers:** *Has daily electricity use per inhabitant in each municipality changed after Norgespris
started (1 October 2025), when we take temperature into account – and in which municipalities is the change largest?*

**Result:** clean Silver tables, a Gold table at grain *municipality × day*, and a simple dashboard with
municipal time series and a before/after comparison.

**Users:** our group and the examiner; possibly analysts interested in the same question.

**Out of scope (on purpose):** single households (EV charging, heat pumps, wood stoves), electricity prices,
real-time/streaming processing.

**Why this scope:** only open data, a clear before/after date, and real Spark work
(hourly → daily aggregation of ~15 million rows per year, joins on municipality and date) that still fits the course server.

**Analysis period:** from **November 2023** (the oldest data Elhub's API still serves – see section 5) to the last complete month.
That gives two winters before Norgespris (2023/24 and 2024/25) and at least one after.

---
# 2. Dataset (source and suitability)

| # | Dataset (folder name in Bronze) | From | Format | One row is | How we get it |
|---|---|---|---|---|---|
| 1 | `consumption_per_group_municipality_hour` | Elhub Energy Data API | CSV `;`, UTF-8 with BOM | kWh and number of metering points per **municipality × hour × consumption group** (household, cabin, business, …) | API CSV export: last ~3 years in one file |
| 2 | `norgespris_consumption_per_group_eac_mba` | Elhub Energy Data API | CSV `,` | kWh **with / without** Norgespris per **day × price area × group × annual-consumption band** | API CSV export (*unverified dataset name – manual file also accepted*) |
| 3 | `norgespris_count_per_municipality_consumption_group` | Elhub open data (data.elhub.no) | CSV | Number of metering points **with Norgespris** per municipality and group | Download, full history (*optional, unverified link*) |
| 4 | `population_07459` | SSB table 07459 | CSV `;` | Population 1 January per **municipality** (one column per year) | SSB PxWebApi v2 |
| 5 | `seklima_daily_mean_temperature` | MET Seklima | CSV `;`, decimal comma | Daily mean temperature per **weather station × day**, all of Norway | Manual export from seklima.met.no |
| 6 | `frost_stations` | MET Frost API | JSON | One weather station with its **municipality** (needed to join weather to municipalities) | API, only if `FROST_CLIENT_ID` is set (*optional*) |

**Why it suits distributed processing:** dataset 1 alone is ~357 municipalities × 8 760 hours × ~5 groups ≈ 15 million rows
per year. Two–three years must be aggregated to days and joined with population, temperature and Norgespris data –
a typical Spark job (shuffle on `groupBy`, joins on municipality and date).

**Licences:** Elhub open data (NLOD), SSB (CC BY 4.0), MET (CC BY 4.0 – stated in the last line of every Seklima file).

**Reproducibility:** every file is kept unchanged in Bronze with source URL, time, size and SHA-256 checksum (section 3).
This matters extra for Elhub: the API only serves the last ~3 years, so **old months disappear from the source** –
our Bronze copy becomes the only copy. Never delete Bronze.

**Known quality risks** (checked with Spark in section 4): different municipality code formats, municipality `0` = "unknown",
broken Norwegian letters in some files, decimal comma and a footer line in Seklima, time zones (Elhub `+01:00/+02:00`,
Seklima "norsk normaltid" = UTC+1 all year), revised Elhub data after month end, only yearly population.

---
# 3. Bronze layer (ingest and preserve)

**Design choices, and why:**

| Choice | Why |
|---|---|
| Keep files **byte-for-byte as downloaded** | Bronze is the proof of what the source delivered. All cleaning is done in Silver, where it can be tested and rerun. |
| Plain Python (`urllib`, `hashlib`, `csv`) for ingestion; Spark only for reading/checking | Downloading files is not a distributed task. Using only the standard library means nothing extra to install on the runner. Spark is used where it adds value (section 4 and Silver). |
| Folder layout `bronze/<source>/<dataset>/<period>/<run_id>__<file>` | Source and dataset visible in the path; `period` is the month (`2025-10`), `snapshot` or `manual`; `run_id` (UTC time of the run) shows *when* it was ingested and keeps old versions side by side. |
| One **manifest** file `bronze/_manifest.csv` | One line per file: run, source URL, file name, time, size, SHA-256, header line, schema status, status. Our traceability *and* monitoring log; easy to open in Excel or Spark. |
| **Skip** a file whose checksum already exists for the dataset | Reruns are safe (idempotent) and never create duplicates. |
| Download the Elhub export again every run | Elhub revises recent data and adds a new month each month. A changed export gets a new checksum and is stored as a new version; Silver uses the newest. An unchanged export is skipped. |
| Compare the **header line** with the previous file of the dataset | A changed header is flagged (`schema_status = changed`) but the file is kept, so Silver decides. No column lists to maintain by hand. |
| `required` flag per source | A failing **required** source makes the run fail (red in GitHub Actions). Optional sources with unverified links only give a warning, so they do not block the pipeline. |
| A `landing/` folder for manual files | Seklima has no open API. Drop the export in `landing/<source>/<dataset>/`; the pipeline copies it into Bronze with the same checks. Works for every dataset, so a broken link never stops the project. |
| Keep old export versions | The export is a *rolling* 3-year window: the oldest month disappears from each new version. Older Bronze versions are the only copy of those months. |
| Secrets only for Frost | Elhub and SSB are open. Frost needs a free client ID, kept in GitHub Secrets as `FROST_CLIENT_ID`. |

### 3.1 Imports
Only the Python standard library is needed for ingestion.

In [ ]:
import base64                                  # build the HTTP Basic-auth header for the Frost API
import csv                                     # read and write the manifest file (handles commas/quotes in text)
import gzip                                    # data.elhub.no sends files gzip-compressed
import hashlib                                 # SHA-256 checksums to recognise identical files
import logging                                 # log messages that also show up in GitHub Actions
import os                                      # environment variables and file paths
import shutil                                  # copy files and delete temp folders
import time                                    # measure run time and wait between download retries
import urllib.request                          # download files over HTTPS (standard library, no pip install needed)
import zipfile                                 # Elhub's ".csv" exports are really ZIP files with a CSV inside
from datetime import datetime, timezone, date  # timestamps (always UTC) and month calculations

logging.basicConfig(level=logging.INFO,                                        # show INFO and above
                    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")  # time - logger - level - text
logger = logging.getLogger("bronze")                                          # named logger for this layer

### 3.2 Configuration: environment and folders
`DAT535_ENV` and `DAT535_OUTPUT_DIR` are set by the workflows. When they are missing (you run the notebook yourself) we fall back to dev in your home folder.

In [ ]:
ENV = os.environ.get("DAT535_ENV", "dev")                     # "dev" or "prod"; default dev when run by hand
DATA_ROOT = os.environ.get("DAT535_OUTPUT_DIR",               # root folder chosen by the workflow ...
                           os.path.expanduser(f"~/dat535-data/{ENV}"))  # ... or ~/dat535-data/dev locally

LANDING_DIR = os.path.join(DATA_ROOT, "landing")              # put manually exported files here
BRONZE_DIR = os.path.join(DATA_ROOT, "bronze")                # raw files, never edited
MANIFEST_PATH = os.path.join(BRONZE_DIR, "_manifest.csv")     # one line per file we have seen

MONTH_RANGE = {                                               # which months to fetch from the Elhub API
    "dev": ("2025-09", "2025-10"),                            # dev: 2 months (only used by "monthly" sources)
    "prod": ("2023-11", None),                                # prod: Nov 2023 -> last complete month (None)
}
REDOWNLOAD_RECENT_MONTHS = 2                                  # always re-fetch the newest months (late/revised data)

### 3.3 Source registry
One dictionary per dataset – **adding a dataset = adding one entry here.**

* `snapshot` – one file with everything, fetched every run (unchanged content is skipped by checksum, so a yearly SSB update is picked up automatically).
  **Elhub's `.csv` export accepts no `startDate`/`endDate`** (the API answers *400 – Path and query parameters are not valid with chosen suffix*).
  Without dates it returns the last ~3 years in one file – the same file you get from api.elhub.no/downloads. So the Elhub datasets are snapshots.
* `monthly` – one request per month with `{start}`/`{end}` in the URL. Not used right now; kept for an API that supports date filters
  (e.g. Elhub's JSON endpoint, if the CSV export becomes too big).
* `manual` – no download; only files placed in `landing/<source>/<dataset>/`.
* `start` – optional first month for a monthly dataset.
* `sep` – the column separator, used when Spark reads the files (section 4 and Silver).
* `required` – if `False`, a failed download is only a warning.

Every dataset **also** picks up files from its landing folder.

⚠️ The URLs could not be tested from where this notebook was written. **Run 3.10 once in dev and look at the manifest (3.11)** – a wrong URL shows up as `failed`.

In [ ]:
ELHUB_API = "https://api.elhub.no/energy-data/v0"             # Elhub Energy Data API (open, no login)

SSB_URL = ("https://data.ssb.no/api/pxwebapi/v2/tables/07459/data?lang=no"      # SSB table 07459, Norwegian labels
           "&valueCodes[Region]=*&codelist[Region]=agg_KommSummer"             # all municipalities, today's structure
           "&outputValues[Region]=aggregated"                                  # sums for merged municipalities
           "&valueCodes[ContentsCode]=*&valueCodes[Tid]=from(2023)"            # population, years 2023 ->
           "&outputFormat=csv&outputFormatParams=separatorsemicolon")          # CSV with ';' like the manual export
# Sex and age are left out of the query -> SSB sums them ("elimination"), so we get total population.

SOURCES = [                                                                    # the datasets the pipeline handles
    {"source": "elhub", "dataset": "consumption_per_group_municipality_hour",  # hourly kWh per municipality and group
     "mode": "snapshot", "sep": ";", "required": True,                   # .csv export = last ~3 years, no date filter
     "url": ELHUB_API + "/municipalities.csv?dataset=CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR"},
    {"source": "elhub", "dataset": "norgespris_consumption_per_group_eac_mba", # kWh with/without Norgespris per price area
     "mode": "snapshot", "sep": ",", "required": False,                  # unverified dataset name -> optional
     "url": ELHUB_API + "/price-areas.csv?dataset=NORGESPRIS_CONSUMPTION_PER_GROUP_EAC_MBA"},
    {"source": "elhub", "dataset": "norgespris_count_per_municipality_consumption_group",  # Norgespris meters per municipality
     "mode": "snapshot", "sep": ",", "required": False,
     "url": "https://data.elhub.no/download/norgespris_count_per_municipality_consumption_group/"
            "norgespris_count_per_municipality_consumption_group-all-en-0000-00-00.csv"},
    {"source": "ssb", "dataset": "population_07459",                          # population per municipality
     "mode": "snapshot", "sep": ";", "required": True, "url": SSB_URL},
    {"source": "met", "dataset": "seklima_daily_mean_temperature",            # Seklima export (manual)
     "mode": "manual", "sep": ";", "required": True},
    {"source": "met", "dataset": "frost_stations",                            # station -> municipality (Frost API)
     "mode": "snapshot", "sep": None, "required": False, "auth": "frost", "ext": "json",
     "url": "https://frost.met.no/sources/v0.jsonld?types=SensorSystem&country=NO"},
]

MANIFEST_COLUMNS = ["run_id", "env", "source", "dataset", "period", "origin",  # columns of _manifest.csv
                    "original_filename", "bronze_path", "ingested_at_utc", "size_bytes",
                    "sha256", "header", "schema_status", "status"]

### 3.4 Helper functions: checksum, header and manifest
* **SHA-256** is a fingerprint of the content: same content → same checksum. Reading in 1 MB blocks means a big file never has to fit in memory.
* **ZIP files:** Elhub's `.csv` export is delivered as a **ZIP** (the file starts with the bytes `PK`) containing Spark-style `part-00000-….csv` files.
  A ZIP stores timestamps, so the ZIP bytes change on every download even when the data is the same. For ZIPs we therefore checksum the **files inside**,
  and read the header from the first CSV inside.
* The **header line** is stored in the manifest to detect schema changes. The BOM (an invisible marker at the start of some files) and line endings are removed – only in the *metadata*, never in the file.
* The **manifest** is only appended to, never rewritten, so it holds the history of all runs.

In [ ]:
def sha256_of(path):
    """Return the SHA-256 checksum of a file."""
    h = hashlib.sha256()                                       # empty checksum object
    with open(path, "rb") as f:                                # binary mode: we hash the exact bytes
        for block in iter(lambda: f.read(1024 * 1024), b""):   # read 1 MB at a time until end of file
            h.update(block)                                    # add the block to the checksum
    return h.hexdigest()                                       # 64-character hex string


def content_sha256(path):
    """Checksum of the data: for a ZIP the files inside (in name order), otherwise the file itself."""
    if not zipfile.is_zipfile(path):                           # normal file (CSV/JSON)
        return sha256_of(path)
    h = hashlib.sha256()
    with zipfile.ZipFile(path) as zf:                          # open the archive
        for name in sorted(zf.namelist()):                     # same order every time
            with zf.open(name) as member:                      # read one file inside the ZIP
                for block in iter(lambda: member.read(1024 * 1024), b""):  # 1 MB at a time
                    h.update(block)
    return h.hexdigest()


def read_header(path):
    """Return the first line of a file, or of the first CSV inside a ZIP (without BOM and line break)."""
    if zipfile.is_zipfile(path):                               # ZIP -> look at the first CSV inside
        with zipfile.ZipFile(path) as zf:
            csvs = sorted(n for n in zf.namelist() if n.lower().endswith(".csv"))  # CSV members only
            if not csvs:                                       # a ZIP without CSV: nothing to compare
                return ""
            with zf.open(csvs[0]) as member:
                first = member.readline(2000)                  # first line of the first CSV
    else:
        with open(path, "rb") as f:                            # binary mode, so odd characters cannot crash us
            first = f.readline(2000)                           # first line only, max 2000 bytes (JSON has long lines)
    return first.decode("utf-8-sig", errors="replace").strip() # utf-8-sig removes the BOM, strip removes \r\n


def read_manifest():
    """Return all rows of the manifest as a list of dicts (empty list on the first run)."""
    if not os.path.exists(MANIFEST_PATH):                      # first run: no manifest yet
        return []
    with open(MANIFEST_PATH, newline="", encoding="utf-8") as f:  # newline="" is required by the csv module
        return list(csv.DictReader(f))                         # one dict per row, keys = column names


def append_manifest(row):
    """Append one row to the manifest (writes the column names if the file is new)."""
    os.makedirs(BRONZE_DIR, exist_ok=True)                     # make sure the bronze folder exists
    is_new = not os.path.exists(MANIFEST_PATH)                 # do we need to write the column names?
    with open(MANIFEST_PATH, "a", newline="", encoding="utf-8") as f:  # "a" = append, never overwrite
        writer = csv.DictWriter(f, fieldnames=MANIFEST_COLUMNS, restval="")  # missing keys become ""
        if is_new:
            writer.writeheader()                               # column names first in a new manifest
        writer.writerow(row)                                   # this file's metadata

### 3.5 Month helpers
Elhub publishes data for a month after the month ends, so the newest month we ask for in prod is the **last complete month**.

In [ ]:
def month_list(start, end):
    """List of 'YYYY-MM' strings from start to end, both included."""
    y, m = map(int, start.split("-"))                          # start year and month as numbers
    ey, em = map(int, end.split("-"))                          # end year and month as numbers
    months = []
    while (y, m) <= (ey, em):                                  # tuples compare year first, then month
        months.append(f"{y:04d}-{m:02d}")                      # e.g. "2025-09"
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)           # next month (December -> January next year)
    return months


def next_month(month):
    """'2025-12' -> '2026-01'."""
    y, m = map(int, month.split("-"))                          # split into numbers
    y, m = (y + 1, 1) if m == 12 else (y, m + 1)               # one month forward
    return f"{y:04d}-{m:02d}"


def last_complete_month(today=None):
    """The month before the current one, e.g. '2026-10' in November 2026."""
    today = today or date.today()                              # allow a fixed date in tests
    y, m = (today.year - 1, 12) if today.month == 1 else (today.year, today.month - 1)  # one month back
    return f"{y:04d}-{m:02d}"

### 3.6 Download with simple retries
Public servers sometimes fail for a moment, so we try 3 times with a growing pause. The response is streamed straight to disk, so a large file never has to fit in memory. `headers` is used for the Frost login.

**gzip:** `data.elhub.no` sends its files gzip-compressed (the file starts with the bytes `1f 8b`). A browser unpacks this automatically, Python's `urllib` does not.
Gzip is only a *packing for the transfer*, so `download()` unpacks it right away – what we store is the CSV exactly as Elhub publishes it.
(ZIP files are different: they are the file Elhub actually publishes, so those are kept as they are – see 3.7.)

In [ ]:
def gunzip_if_needed(path):
    """If the file is gzip-compressed, replace it with the unpacked content."""
    with open(path, "rb") as f:
        if f.read(2) != b"\x1f\x8b":                         # gzip files always start with these two bytes
            return                                             # not gzip -> nothing to do
    with gzip.open(path, "rb") as src, open(path + ".unpacked", "wb") as dst:  # read compressed, write plain
        shutil.copyfileobj(src, dst)                           # stream, so big files are fine
    os.replace(path + ".unpacked", path)                       # swap the unpacked file in
    logger.info(f"  unpacked gzip: {os.path.basename(path)}")


def download(url, target_path, headers=None, attempts=3):
    """Download url to target_path; retry a few times, raise the last error if all attempts fail."""
    all_headers = {"User-Agent": "DAT535-student-project"}     # say politely who we are
    all_headers.update(headers or {})                          # add e.g. an Authorization header
    for attempt in range(1, attempts + 1):                     # attempt 1, 2, 3
        try:
            req = urllib.request.Request(url, headers=all_headers)                    # build the request
            with urllib.request.urlopen(req, timeout=300) as resp, open(target_path, "wb") as out:  # max 5 min
                shutil.copyfileobj(resp, out)                  # stream the response to the file
            gunzip_if_needed(target_path)                      # undo gzip transfer compression (data.elhub.no)
            return                                             # success -> stop retrying
        except Exception as e:                                 # network error, HTTP 400/404/500, timeout ...
            logger.warning(f"Download attempt {attempt}/{attempts} failed for {url}: {e}")
            if attempt == attempts:                            # the last attempt also failed ...
                raise                                          # ... let the caller record the failure
            time.sleep(5 * attempt)                            # wait 5 s, then 10 s


def auth_headers(source_cfg):
    """Extra HTTP headers for sources that need a login (only Frost). None = cannot download."""
    if source_cfg.get("auth") != "frost":                      # all other sources are open
        return {}
    client_id = os.environ.get("FROST_CLIENT_ID")              # from GitHub Secrets or your shell
    if not client_id:                                          # no ID -> skip this source
        return None
    token = base64.b64encode(f"{client_id}:".encode()).decode()  # Basic auth: "<client_id>:" (empty password)
    return {"Authorization": f"Basic {token}"}

### 3.7 Ingest one file into Bronze
The heart of the Bronze layer. For each file:
1. compute checksum and read the header,
2. **duplicate?** same checksum already stored for this dataset → record `skipped_duplicate` and stop,
3. **schema check** against the newest earlier file of the dataset → `first_file`, `ok` or `changed`,
4. copy the file unchanged to `bronze/<source>/<dataset>/<period>/<run_id>__<filename>`;
   if it is a **ZIP**, it is saved with the `.zip` ending and *also* unpacked into the folder `<run_id>__<name>_unzipped/` next to it.
   The ZIP stays the untouched original; the unpacked copy is only there so Spark can read it (Spark cannot read ZIP files),
5. write one manifest line.

`state` is a small dict with the run ID, all manifest rows so far and counters for this run.

In [ ]:
def ingest_file(state, src_path, source, dataset, period, origin, original_filename):
    """Copy one file unchanged into Bronze unless an identical file is already there."""
    checksum = content_sha256(src_path)                         # fingerprint of the content (inside a ZIP: the files)
    header = read_header(src_path)                              # column-names line
    earlier = [r for r in state["manifest"]                     # earlier files of the SAME dataset ...
               if r["dataset"] == dataset and r["status"] == "ingested"]  # ... that were actually stored

    row = {"run_id": state["run_id"], "env": ENV, "source": source, "dataset": dataset,  # metadata for the manifest
           "period": period, "origin": origin, "original_filename": original_filename,
           "ingested_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "size_bytes": os.path.getsize(src_path), "sha256": checksum, "header": header}

    if any(r["sha256"] == checksum for r in earlier):           # exact same content seen before?
        row["status"] = "skipped_duplicate"                     # yes -> do not store it again
        state["counts"]["skipped_duplicate"] += 1
        logger.info(f"  skip (already ingested): {dataset} {period}")
    else:
        if not earlier:                                         # first file ever for this dataset
            row["schema_status"] = "first_file"
        elif earlier[-1]["header"] == header:                   # same columns as the newest earlier file
            row["schema_status"] = "ok"
        else:                                                   # columns changed -> keep the file, but flag it
            row["schema_status"] = "changed"
            logger.warning(f"  HEADER CHANGED for {dataset}: check before Silver!")
        is_zip = zipfile.is_zipfile(src_path)                                 # Elhub exports are ZIPs
        if is_zip and not original_filename.lower().endswith(".zip"):         # give it the correct file ending
            original_filename = os.path.splitext(original_filename)[0] + ".zip"
        target_dir = os.path.join(BRONZE_DIR, source, dataset, period)        # e.g. bronze/elhub/<ds>/snapshot
        os.makedirs(target_dir, exist_ok=True)                                # create the folder if missing
        target = os.path.join(target_dir, f"{state['run_id']}__{original_filename}")  # run_id keeps versions apart
        shutil.copy2(src_path, target)                                        # byte-for-byte copy of the original
        if is_zip:                                                            # unpack a copy for Spark
            with zipfile.ZipFile(target) as zf:
                zf.extractall(target[:-len(".zip")] + "_unzipped")            # <run_id>__<name>_unzipped/part-....csv
        row["bronze_path"] = os.path.relpath(target, BRONZE_DIR)              # path relative to bronze/
        row["status"] = "ingested"
        state["counts"]["ingested"] += 1
        logger.info(f"  ingested: {row['bronze_path']}")

    append_manifest(row)                                        # write the line to disk right away
    state["manifest"].append(row)                               # and remember it for the rest of the run

### 3.8 Download → ingest, and which months to fetch
* A download goes to a temporary file first, so only complete files are ingested. A failure is written to the manifest (`failed: ...`) and the run continues.
* For monthly datasets (none at the moment) we only fetch months that are **missing** in Bronze, plus the **2 newest months** again (late/revised data).

In [ ]:
def download_and_ingest(state, cfg, url, period, filename):
    """Download url to a temp file and ingest it; record failures in the manifest."""
    source, dataset = cfg["source"], cfg["dataset"]
    headers = auth_headers(cfg)                                  # {} for open sources, None if login is missing
    if headers is None:                                          # e.g. FROST_CLIENT_ID not set
        logger.info(f"  skip {dataset}: no credentials (set FROST_CLIENT_ID to enable)")
        return
    tmp_path = os.path.join(state["tmp_dir"], filename)          # temp file in bronze/_tmp/<run_id>/
    try:
        download(url, tmp_path, headers)                         # may raise after 3 failed attempts
        ingest_file(state, tmp_path, source, dataset, period, url, filename)  # checks + copy + manifest
    except Exception as e:                                       # download (or copy) failed
        level = "failed" if cfg["required"] else "warning"       # optional sources do not fail the run
        state["counts"][level] += 1
        append_manifest({"run_id": state["run_id"], "env": ENV, "source": source, "dataset": dataset,
                         "period": period, "origin": url, "original_filename": filename,
                         "ingested_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                         "status": f"{level}: {type(e).__name__} {e}"[:200]})  # e.g. "failed: HTTPError ..."
        logger.error(f"  {level.upper()}: {url} ({e})")
    finally:
        if os.path.exists(tmp_path):                             # always clean up the temp file
            os.remove(tmp_path)


def months_to_download(state, cfg):
    """Months missing in Bronze, plus the newest months again (to catch revised data)."""
    start, end = MONTH_RANGE[ENV]                                # range for dev or prod
    start = max(start, cfg.get("start", start))                  # some datasets start later ("YYYY-MM" sorts as text)
    months = month_list(start, end or last_complete_month())     # None = up to the last complete month
    have = {r["period"] for r in state["manifest"]               # months we already stored
            if r["dataset"] == cfg["dataset"] and r["status"] == "ingested"}
    recent = set(months[-REDOWNLOAD_RECENT_MONTHS:])             # the newest N months
    return [m for m in months if m not in have or m in recent]   # missing OR recent

### 3.9 The whole Bronze run
Loops over the source registry and returns `status` and `elapsed_time` – `run_pipeline.py` turns `failed` into exit code 1, which shows as a red run in GitHub Actions.

In [ ]:
def run_bronze():
    """Run Bronze ingestion for all sources and return a summary dict."""
    start_time = time.time()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")  # e.g. 20261109T201500Z
    state = {"run_id": run_id,                                 # shared state for the helper functions
             "tmp_dir": os.path.join(BRONZE_DIR, "_tmp", run_id),
             "manifest": read_manifest(),                      # everything ingested in earlier runs
             "counts": {"ingested": 0, "skipped_duplicate": 0, "warning": 0, "failed": 0}}
    os.makedirs(state["tmp_dir"], exist_ok=True)
    logger.info(f"Bronze run {run_id} (env={ENV}) -> {BRONZE_DIR}")

    for cfg in SOURCES:                                        # one dataset at a time
        source, dataset, mode = cfg["source"], cfg["dataset"], cfg["mode"]
        logger.info(f"[{source}/{dataset}] mode={mode}")
        ext = cfg.get("ext", "csv")                            # file extension (Frost gives JSON)

        if mode == "monthly":                                  # one API request per month
            for month in months_to_download(state, cfg):
                url = cfg["url"].format(start=f"{month}-01",                 # first day of the month
                                        end=f"{next_month(month)}-01")       # first day of next month
                download_and_ingest(state, cfg, url, month, f"{dataset}_{month}.{ext}")
        elif mode == "snapshot":                               # one file with everything
            download_and_ingest(state, cfg, cfg["url"], "snapshot", f"{dataset}.{ext}")

        landing = os.path.join(LANDING_DIR, source, dataset)   # manual files for this dataset
        os.makedirs(landing, exist_ok=True)                    # create it so you can see where files go
        for name in sorted(os.listdir(landing)):               # every file in the landing folder
            if name.lower().endswith((".csv", ".json", ".zip")):  # only data files
                ingest_file(state, os.path.join(landing, name), source, dataset,
                            "manual", f"landing/{source}/{dataset}", name)  # duplicates skipped by checksum

    shutil.rmtree(state["tmp_dir"], ignore_errors=True)        # remove the temp folder
    elapsed = time.time() - start_time
    logger.info(f"Bronze done in {elapsed:.1f}s: {state['counts']}")
    status = "success" if state["counts"]["failed"] == 0 else "failed"  # a failed REQUIRED source -> failed run
    return {"status": status, "elapsed_time": elapsed, **state["counts"]}

### 3.10 Run it

**Put the manual files in the landing folders first** (the next cell creates them and prints the paths):

| File | Copy it to |
|---|---|
| Seklima export(s), all stations, from 2023-11-01 – several files are fine | `landing/met/seklima_daily_mean_temperature/` |
| `norgesprisConsumptionPerGroupEacMba_0001.csv` (in case the API name is wrong) | `landing/elhub/norgespris_consumption_per_group_eac_mba/` |
| `consumptionPerGroupMunicipalityHour-...csv` – **only** if the API download fails (it is the same export) | `landing/elhub/consumption_per_group_municipality_hour/` |

Running the cell twice is safe: the second run downloads again but skips every unchanged file as `skipped_duplicate`.
The first Elhub consumption download is large (3 years of hourly data) – expect it to take a while.

In [ ]:
for cfg in SOURCES:                                            # show where each dataset's manual files go
    path = os.path.join(LANDING_DIR, cfg["source"], cfg["dataset"])  # landing/<source>/<dataset>
    os.makedirs(path, exist_ok=True)                           # create the folder now
    print(f"{cfg['mode']:9s} {path}")                          # mode + folder

In [ ]:
result = run_bronze()                                          # run the whole Bronze layer
result                                                         # show the summary

### 3.11 The manifest (traceability and monitoring)
Evidence for section 3 of the definition: where each file came from, when, its checksum, and whether it was new, a duplicate, a schema change or a failure. The manifest is tiny, so pandas is enough here.

In [ ]:
import pandas as pd                                            # small table -> pandas is fine

manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")   # read everything as text
cols = ["run_id", "dataset", "period", "status", "schema_status", "size_bytes"]  # most useful columns
display(manifest[cols].tail(20))                               # newest 20 lines
print(manifest.groupby(["run_id", "status"]).size())           # files per run and status

---
# 4. Check Bronze with Spark and profile the data (evidence for section 2)

This proves the Bronze files can be read by Spark and measures what Silver will have to fix.
Every column is read **as text**: Bronze/profiling must not guess types – type conversion is a Silver decision.

### 4.1 SparkSession
Small, explicit settings: adaptive query execution on, 8 shuffle partitions (the default of 200 is far too many for this data size), 2 GB driver memory.

In [ ]:
from pyspark.sql import SparkSession                            # entry point to Spark
from pyspark.sql import functions as F                          # column functions (count, min, max, ...)

spark = (SparkSession.builder
         .appName("DAT535-Group24-Bronze-Check")                # name in the Spark UI
         .config("spark.sql.adaptive.enabled", "true")          # let Spark tune partitions/joins at runtime
         .config("spark.sql.shuffle.partitions", "8")           # small data -> few shuffle partitions
         .config("spark.driver.memory", "2g")                   # enough for profiling
         .getOrCreate())                                        # reuse a running session if there is one
spark.sparkContext.setLogLevel("WARN")                          # hide Spark's INFO noise
print(spark.version)

### 4.2 Read the newest Bronze version of each dataset
For a dataset with many monthly files we pass the whole list to Spark, which reads them in parallel.
For a ZIP we read its unpacked folder (`…_unzipped/`); Spark reads every CSV in a folder and skips files starting with `_` or `.`.
The separator is taken from the stored header line (more `;` than `,` → `;`), because the CSV inside Elhub's ZIP may differ from the files we downloaded by hand. We only remove the BOM character from the *column names* so we can refer to them – the data itself is not touched.

In [ ]:
def newest_files(dataset):
    """Absolute paths of the newest ingested file per period of a dataset."""
    rows = manifest[(manifest.dataset == dataset) & (manifest.status == "ingested")]  # stored files only
    newest = rows.sort_values("run_id").groupby("period").tail(1)                     # newest version per period
    paths = [os.path.join(BRONZE_DIR, p) for p in newest.bronze_path]                 # full paths
    return [p[:-len(".zip")] + "_unzipped" if p.endswith(".zip") else p for p in paths]  # ZIP -> its unpacked folder

def guess_sep(dataset, default):
    """Separator from the header line stored in the manifest: ';' if it has more ';' than ','."""
    rows = manifest[(manifest.dataset == dataset) & (manifest.status == "ingested")]
    if rows.empty:                                              # nothing stored -> use the registry value
        return default
    header = rows.sort_values("run_id").header.iloc[-1]         # header of the newest file
    return ";" if header.count(";") > header.count(",") else ","

def read_bronze(cfg):
    """Read a Bronze CSV dataset with Spark, all columns as text; None if nothing to read."""
    files = newest_files(cfg["dataset"])                        # which files/folders to read
    if not files or cfg["sep"] is None:                         # nothing ingested, or not a CSV (Frost JSON)
        print(f"-- {cfg['dataset']}: nothing to read as CSV")
        return None
    sep = guess_sep(cfg["dataset"], cfg["sep"])                 # Elhub's ZIP may use another separator than our samples
    df = (spark.read
          .option("header", True)                               # first line = column names
          .option("inferSchema", False)                         # keep everything as text in Bronze
          .option("sep", sep)                                   # separator for this dataset
          .option("encoding", "UTF-8")
          .csv(files))                                          # list of files -> one DataFrame
    return df.toDF(*[c.replace("﻿", "").strip() for c in df.columns])  # remove BOM from column names only

bronze = {cfg["dataset"]: read_bronze(cfg) for cfg in SOURCES}  # one DataFrame per dataset

### 4.3 Row counts and columns
`count()` is an *action*: this is where Spark actually reads the files. These numbers go into the report as the Bronze row counts that Silver results are compared with.

In [ ]:
for name, df in bronze.items():                                 # every dataset
    if df is None:                                              # skip what is not ingested / not CSV
        continue
    print(f"\n=== {name}: {df.count():,} rows, {len(df.columns)} columns")  # action -> Spark reads the files
    print(df.columns)                                           # column names
    df.show(3, truncate=False)                                  # first 3 rows

### 4.4 Source-specific checks that Silver must handle
One small query per known risk. Each printed number is a finding for the report.

In [ ]:
cons = bronze["consumption_per_group_municipality_hour"]        # Elhub hourly consumption
if cons is not None:
    cons.agg(F.min("START_TIME").alias("first_hour"),           # period covered by the files
             F.max("START_TIME").alias("last_hour"),
             F.countDistinct("MUNICIPALITY_NUMBER").alias("municipalities")).show(truncate=False)
    cons.groupBy("CONSUMPTION_GROUP").count().show()            # rows per consumption group
    print("Rows with municipality 0 ('unknown'):",
          cons.filter(F.col("MUNICIPALITY_NUMBER") == "0").count())       # cannot be joined to SSB -> handle in Silver
    print("Names with broken letters (√ instead of æøå):",
          cons.filter(F.col("NAME").contains("√")).select("NAME").distinct().count())  # encoding check
    print("Municipality numbers shorter than 4 digits:",
          cons.filter(F.length("MUNICIPALITY_NUMBER") < 4).select("MUNICIPALITY_NUMBER").distinct().count())

In [ ]:
met = bronze["seklima_daily_mean_temperature"]                  # Seklima temperatures
if met is not None:
    station_col, temp_col = met.columns[1], met.columns[-1]     # "Stasjon" and the temperature column
    print("Footer/metadata lines (no station id):",
          met.filter(F.col(station_col).isNull() | (F.col(station_col) == "")).count())  # "Data er gyldig per ..."
    print("Missing temperature ('-' or empty):",
          met.filter(F.col(temp_col).isNull() | (F.col(temp_col) == "-")).count())
    print("Values with decimal comma:", met.filter(F.col(temp_col).contains(",")).count())  # -> '.' in Silver
    print("Number of stations:", met.select(station_col).distinct().count())               # coverage of the export

In [ ]:
pop = bronze["population_07459"]                                # SSB population
if pop is not None:
    region = pop.columns[0]                                     # region column ("Kommune"/"region")
    pop.select(region).show(3, truncate=False)                  # codes look like "K-3101 Halden"
    print("Rows that are not real municipalities:",
          pop.filter(F.col(region).rlike("Rest|Kontinentalsokkelen|[Uu]oppgitt")).count())  # remove in Silver

---
# 5. Problems found and urgent improvements

We checked the definition guide v3 against the real files. Ordered by how much they matter.

### 🔴 Must fix before / in Silver

1. **Elhub's API only serves the last ~3 years (rolling).** The file name `…2023-11-08-to-2026-11-08` shows it, and the API guidelines describe a "max age from now". The definition's period *January 2014 – October 2026* is not possible.
   → Analysis period **Nov 2023 → last complete month** (two winters before Norgespris, one after). Bronze keeps every export version, because each new version has lost its oldest month.
   → The `.csv` export has **no date filter** (HTTP 400 if you add `startDate`/`endDate`), so each download is the full 3 years. Watch the disk on the server: every month adds one new copy. Once Silver has turned a version into Parquet, older Bronze copies *can* be deleted by hand – but only after Silver has saved their oldest months.

2. **Weather is per station, not per municipality** (the definition says "per metering point" – should be *per weather station*). Seklima rows have no municipality, so they cannot be joined to Elhub/SSB directly.
   → Station → municipality reference. Easiest: MET's Frost API (`frost_stations` source, needs a free client ID from frost.met.no, stored as GitHub secret `FROST_CLIENT_ID`). Municipalities without a station get the temperature of the nearest station (or the county average) in Silver – decide which and document it.

3. **Use Elhub's Norgespris count per municipality instead of building our own distribution key** (if the optional `norgespris_count_per_municipality_consumption_group` download works). It gives the Norgespris share per municipality directly – much simpler and more accurate than spreading price-area totals down to municipalities.

4. **Municipality `0` = "Ukjent tilhørighet" (unknown) in the consumption data.** It has real kWh but no population. → Keep it out of per-person figures, but report how much kWh it is.

### 🟡 Important for Silver design

5. **Different municipality codes.** SSB: `K-3101 Halden` (prefix + code + name). Elhub: `0301`, or a short number like `0`. The 2024 county split changed many codes, and data from 2023 may use old ones.
   → Silver: one `municipality_no` as a 4-character string (`lpad`), plus a small old → new code mapping table.

6. **Don't convert daily data to UTC before grouping.** Elhub uses Norwegian local time with `+01:00/+02:00`; Seklima uses *norsk normaltid* (UTC+1 all year). Grouping UTC hours by day shifts every day by 1–2 hours.
   → Build the **date in Norwegian local time** for the daily grain. Keep technical timestamps (`ingested_at`, `LAST_UPDATE_TIME`) in UTC.

7. **Broken Norwegian letters** in the consumption sample (`tilh√∏righet` instead of `tilhørighet`). Probably the file was opened and saved again on a Mac. Check the API download in the manifest/Spark check (4.4). Either way: join on **numbers**, never on names.

8. **Compression:** the Elhub API's ".csv" export is a **ZIP** (kept, plus an unpacked copy); `data.elhub.no` sends **gzip**, which `download()` unpacks.
   **Elhub's ".csv" export is a ZIP** (with Spark `part-…csv` files inside), not a plain CSV. Bronze keeps the ZIP and an unpacked copy; Silver reads the unpacked folder.
   Inside the ZIP each `part` file has its own header line – Spark's `header=True` handles that per file.

9. **Separators differ per dataset:** consumption `;`, Norgespris EAC file `,`, SSB `;`, Seklima `;` with decimal comma, date `dd.mm.yyyy`, last line is a licence text, missing values `-`. Already captured in the `sep` field of the registry.

10. **SSB is wide** (one column per year) → unpivot to `municipality, year, population` in Silver, and drop non-municipality rows (`K-23 Kontinentalsokkelen`, `K-Rest …`).

11. **Revised and overlapping data.** Each new export overlaps the previous ones by almost 3 years, with revised values for recent months. → Silver keeps one row per (hour, municipality, group): the one with the newest `LAST_UPDATE_TIME`.

### 🟢 Small corrections to the definition text

12. Bronze: GitHub Secrets are only needed for Frost; Elhub and SSB are open.
13. Gold "Upstream failure: Data in Gold are published" → say *what* happens: "if Silver quality checks fail, Gold is not rebuilt and the previous version stays published".
14. Seklima export limits: exporting all stations for 3 years may need several exports (e.g. one per year). The landing folder accepts any number of files.

---
# 6. How to run (summary)

| Where | Command |
|---|---|
| Notebook (dev) | Run all cells; data in `~/dat535-data/dev/` |
| Terminal on the server | `source ~/spark-env/bin/activate && python run_pipeline.py bronze` |
| Tests | `pytest -q tests/` |
| GitHub Actions – dev | push to `dev` → `dat535-dev.yml`: tests, then Bronze |
| GitHub Actions – prod | merge to `main` (+ approval), monthly schedule, or manual run → `dat535-prod.yml` |

The self-hosted runner needs internet access to `api.elhub.no`, `data.elhub.no`, `data.ssb.no` (and `frost.met.no`).
Manual files go into the landing folders **on the server**: `~/dat535-data/<dev|prod>/landing/…`.

**Next notebook (02):** Silver – read the newest Bronze files with explicit schemas, clean, standardise and write Parquet.